{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 현대자동차 도장 검사 기반 생산량 통계 분석\n",
    "\n",
    "## 데이터 출처\n",
    "- **데이터셋**: 현대 오토에버 Track A - 자동차 도장 품질 검사 데이터\n",
    "- **기간**: 2023-01-01 ~ 2025-01-24 (약 25개월, 755일)\n",
    "- **총 검사 건수**: 3,000,000건\n",
    "- **분석 기준**: `daily_summary.csv` (318,812행) - 일별/공장/라인/모델/교대조 집계 데이터"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.ticker as mticker\n",
    "import numpy as np\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# 한글 폰트 설정\n",
    "plt.rcParams['font.family'] = 'Malgun Gothic'\n",
    "plt.rcParams['axes.unicode_minus'] = False\n",
    "plt.rcParams['figure.figsize'] = (14, 6)\n",
    "plt.rcParams['figure.dpi'] = 100\n",
    "\n",
    "# 데이터 경로\n",
    "DATA_DIR = 'dataset/현대_오토에버/track_a_data/'\n",
    "\n",
    "print('라이브러리 로딩 완료')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 1. 데이터 로딩\n",
    "# ========================================\n",
    "daily = pd.read_csv(f'{DATA_DIR}daily_summary.csv', parse_dates=['date'])\n",
    "master_model = pd.read_csv(f'{DATA_DIR}master_model.csv')\n",
    "master_plant = pd.read_csv(f'{DATA_DIR}master_plant_line.csv')\n",
    "master_color = pd.read_csv(f'{DATA_DIR}master_color.csv')\n",
    "\n",
    "# 마스터 테이블 조인\n",
    "daily = daily.merge(master_model[['model_code', 'model_name', 'brand']], on='model_code', how='left')\n",
    "daily = daily.merge(master_plant[['plant_code', 'plant_name', 'line_code']].drop_duplicates(), \n",
    "                    on=['plant_code', 'line_code'], how='left')\n",
    "\n",
    "# 파생 컬럼 추가\n",
    "daily['year'] = daily['date'].dt.year\n",
    "daily['month'] = daily['date'].dt.month\n",
    "daily['year_month'] = daily['date'].dt.to_period('M')\n",
    "daily['weekday'] = daily['date'].dt.dayofweek  # 0=월, 6=일\n",
    "daily['weekday_name'] = daily['date'].dt.day_name()\n",
    "\n",
    "print(f'daily_summary 로딩 완료: {daily.shape}')\n",
    "print(f'기간: {daily[\"date\"].min().date()} ~ {daily[\"date\"].max().date()}')\n",
    "print(f'총 검사 건수: {daily[\"total_inspections\"].sum():,}건')\n",
    "daily.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 2. 전체 생산량 요약 (Overview)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 2. 전체 생산량 요약\n",
    "# ========================================\n",
    "total = daily['total_inspections'].sum()\n",
    "total_pass = daily['pass_count'].sum()\n",
    "total_fail = daily['fail_count'].sum()\n",
    "total_days = daily['date'].nunique()\n",
    "date_range = f\"{daily['date'].min().date()} ~ {daily['date'].max().date()}\"\n",
    "\n",
    "overview = pd.DataFrame({\n",
    "    '항목': ['분석 기간', '총 운영일수', '총 검사(생산) 건수', \n",
    "            '합격(PASS) 건수', '불합격(FAIL) 건수', '전체 양품률',\n",
    "            '일 평균 생산량', '일 최대 생산량', '일 최소 생산량'],\n",
    "    '값': [date_range, f'{total_days}일',\n",
    "           f'{total:,}대', f'{total_pass:,}대', f'{total_fail:,}대',\n",
    "           f'{total_pass/total*100:.2f}%',\n",
    "           f'{total/total_days:,.0f}대/일',\n",
    "           f\"{daily.groupby('date')['total_inspections'].sum().max():,}대/일\",\n",
    "           f\"{daily.groupby('date')['total_inspections'].sum().min():,}대/일\"]\n",
    "})\n",
    "\n",
    "print('=' * 50)\n",
    "print('      현대자동차 생산량 전체 요약')\n",
    "print('=' * 50)\n",
    "for _, row in overview.iterrows():\n",
    "    print(f'  {row[\"항목\"]:20s} : {row[\"값\"]}')\n",
    "print('=' * 50)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 3. 연도별 생산량 추이"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 3. 연도별 생산량\n",
    "# ========================================\n",
    "yearly = daily.groupby('year').agg(\n",
    "    총생산량=('total_inspections', 'sum'),\n",
    "    합격=('pass_count', 'sum'),\n",
    "    불합격=('fail_count', 'sum'),\n",
    "    운영일수=('date', 'nunique')\n",
    ").reset_index()\n",
    "yearly['양품률(%)'] = (yearly['합격'] / yearly['총생산량'] * 100).round(2)\n",
    "yearly['일평균생산량'] = (yearly['총생산량'] / yearly['운영일수']).astype(int)\n",
    "\n",
    "print('\\n[ 연도별 생산량 ]')\n",
    "display(yearly.style.format({\n",
    "    '총생산량': '{:,.0f}', '합격': '{:,.0f}', '불합격': '{:,.0f}',\n",
    "    '일평균생산량': '{:,.0f}', '양품률(%)': '{:.2f}%'\n",
    "}))\n",
    "\n",
    "# 시각화\n",
    "fig, ax1 = plt.subplots(figsize=(10, 5))\n",
    "bars = ax1.bar(yearly['year'].astype(str), yearly['총생산량'], \n",
    "               color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.8, width=0.5)\n",
    "ax1.set_ylabel('생산량 (대)', fontsize=12)\n",
    "ax1.set_xlabel('연도', fontsize=12)\n",
    "ax1.set_title('연도별 총 생산량', fontsize=14, fontweight='bold')\n",
    "for bar, val in zip(bars, yearly['총생산량']):\n",
    "    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10000,\n",
    "             f'{val:,.0f}', ha='center', fontsize=11, fontweight='bold')\n",
    "ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/10000:.0f}만'))\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 4. 월별 생산량 추이"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 4. 월별 생산량 추이 (시계열)\n",
    "# ========================================\n",
    "monthly = daily.groupby('year_month').agg(\n",
    "    총생산량=('total_inspections', 'sum'),\n",
    "    합격=('pass_count', 'sum'),\n",
    "    불합격=('fail_count', 'sum'),\n",
    "    운영일수=('date', 'nunique')\n",
    ").reset_index()\n",
    "monthly['양품률(%)'] = (monthly['합격'] / monthly['총생산량'] * 100).round(2)\n",
    "monthly['불량률(%)'] = (monthly['불합격'] / monthly['총생산량'] * 100).round(2)\n",
    "monthly['x_label'] = monthly['year_month'].astype(str)\n",
    "\n",
    "fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))\n",
    "\n",
    "# 상단: 월별 생산량\n",
    "ax1.fill_between(range(len(monthly)), monthly['총생산량'], alpha=0.3, color='#1f77b4')\n",
    "ax1.plot(range(len(monthly)), monthly['총생산량'], '-o', color='#1f77b4', \n",
    "         markersize=4, linewidth=1.5, label='총 생산량')\n",
    "ax1.set_ylabel('생산량 (대)', fontsize=12)\n",
    "ax1.set_title('월별 생산량 추이', fontsize=14, fontweight='bold')\n",
    "ax1.set_xticks(range(0, len(monthly), 2))\n",
    "ax1.set_xticklabels(monthly['x_label'].iloc[::2], rotation=45, ha='right', fontsize=8)\n",
    "ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/10000:.1f}만'))\n",
    "ax1.legend()\n",
    "ax1.grid(axis='y', alpha=0.3)\n",
    "\n",
    "# 하단: 월별 불량률\n",
    "ax2.plot(range(len(monthly)), monthly['불량률(%)'], '-s', color='#d62728', \n",
    "         markersize=4, linewidth=1.5, label='불량률')\n",
    "ax2.axhline(y=monthly['불량률(%)'].mean(), color='gray', linestyle='--', alpha=0.7,\n",
    "            label=f'평균 불량률 ({monthly[\"불량률(%)\"].mean():.2f}%)')\n",
    "ax2.set_ylabel('불량률 (%)', fontsize=12)\n",
    "ax2.set_xlabel('월', fontsize=12)\n",
    "ax2.set_title('월별 불량률 추이', fontsize=14, fontweight='bold')\n",
    "ax2.set_xticks(range(0, len(monthly), 2))\n",
    "ax2.set_xticklabels(monthly['x_label'].iloc[::2], rotation=45, ha='right', fontsize=8)\n",
    "ax2.legend()\n",
    "ax2.grid(axis='y', alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(f'\\n월 평균 생산량: {monthly[\"총생산량\"].mean():,.0f}대')\n",
    "print(f'월 최대 생산량: {monthly[\"총생산량\"].max():,.0f}대 ({monthly.loc[monthly[\"총생산량\"].idxmax(), \"x_label\"]})')\n",
    "print(f'월 최소 생산량: {monthly[\"총생산량\"].min():,.0f}대 ({monthly.loc[monthly[\"총생산량\"].idxmin(), \"x_label\"]})')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 5. 공장별 생산량"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 5. 공장별 생산량\n",
    "# ========================================\n",
    "by_plant = daily.groupby(['plant_code', 'plant_name']).agg(\n",
    "    총생산량=('total_inspections', 'sum'),\n",
    "    합격=('pass_count', 'sum'),\n",
    "    불합격=('fail_count', 'sum'),\n",
    "    라인수=('line_code', 'nunique')\n",
    ").reset_index().sort_values('총생산량', ascending=False)\n",
    "by_plant['양품률(%)'] = (by_plant['합격'] / by_plant['총생산량'] * 100).round(2)\n",
    "by_plant['점유율(%)'] = (by_plant['총생산량'] / by_plant['총생산량'].sum() * 100).round(2)\n",
    "\n",
    "print('\\n[ 공장별 생산량 ]')\n",
    "display(by_plant.style.format({\n",
    "    '총생산량': '{:,.0f}', '합격': '{:,.0f}', '불합격': '{:,.0f}',\n",
    "    '양품률(%)': '{:.2f}%', '점유율(%)': '{:.2f}%'\n",
    "}))\n",
    "\n",
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "# 막대그래프\n",
    "colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']\n",
    "bars = ax1.barh(by_plant['plant_name'], by_plant['총생산량'], color=colors, alpha=0.85)\n",
    "for bar, val in zip(bars, by_plant['총생산량']):\n",
    "    ax1.text(val + 5000, bar.get_y() + bar.get_height()/2,\n",
    "             f'{val:,.0f}대', va='center', fontsize=11, fontweight='bold')\n",
    "ax1.set_xlabel('생산량 (대)', fontsize=12)\n",
    "ax1.set_title('공장별 총 생산량', fontsize=14, fontweight='bold')\n",
    "ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/10000:.0f}만'))\n",
    "\n",
    "# 파이차트\n",
    "ax2.pie(by_plant['총생산량'], labels=by_plant['plant_name'], autopct='%1.1f%%',\n",
    "        colors=colors, startangle=90, textprops={'fontsize': 11})\n",
    "ax2.set_title('공장별 생산 비중', fontsize=14, fontweight='bold')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 6. 생산 라인별 생산량"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 6. 생산 라인별 생산량\n",
    "# ========================================\n",
    "by_line = daily.groupby(['plant_name', 'line_code']).agg(\n",
    "    총생산량=('total_inspections', 'sum'),\n",
    "    합격=('pass_count', 'sum'),\n",
    "    불합격=('fail_count', 'sum')\n",
    ").reset_index().sort_values('총생산량', ascending=True)\n",
    "by_line['양품률(%)'] = (by_line['합격'] / by_line['총생산량'] * 100).round(2)\n",
    "by_line['라인명'] = by_line['plant_name'] + ' - ' + by_line['line_code']\n",
    "\n",
    "# 공장별 색상 매핑\n",
    "plant_colors = {'울산공장': '#2196F3', '아산공장': '#FF9800', \n",
    "                '광주공장': '#4CAF50', '화성공장': '#9C27B0'}\n",
    "bar_colors = [plant_colors.get(p, 'gray') for p in by_line['plant_name']]\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(14, 7))\n",
    "bars = ax.barh(by_line['라인명'], by_line['총생산량'], color=bar_colors, alpha=0.85)\n",
    "for bar, val in zip(bars, by_line['총생산량']):\n",
    "    ax.text(val + 2000, bar.get_y() + bar.get_height()/2,\n",
    "            f'{val:,.0f}', va='center', fontsize=9)\n",
    "ax.set_xlabel('생산량 (대)', fontsize=12)\n",
    "ax.set_title('생산 라인별 총 생산량', fontsize=14, fontweight='bold')\n",
    "ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/10000:.0f}만'))\n",
    "\n",
    "# 범례\n",
    "from matplotlib.patches import Patch\n",
    "legend_handles = [Patch(facecolor=c, label=p) for p, c in plant_colors.items()]\n",
    "ax.legend(handles=legend_handles, loc='lower right', fontsize=10)\n",
    "ax.grid(axis='x', alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print('\\n[ 라인별 상세 ]')\n",
    "display(by_line[['라인명', '총생산량', '합격', '불합격', '양품률(%)']].sort_values('총생산량', ascending=False).style.format({\n",
    "    '총생산량': '{:,.0f}', '합격': '{:,.0f}', '불합격': '{:,.0f}', '양품률(%)': '{:.2f}%'\n",
    "}))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 7. 차종(모델)별 생산량"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 7. 차종(모델)별 생산량\n",
    "# ========================================\n",
    "by_model = daily.groupby(['model_code', 'model_name', 'brand']).agg(\n",
    "    총생산량=('total_inspections', 'sum'),\n",
    "    합격=('pass_count', 'sum'),\n",
    "    불합격=('fail_count', 'sum')\n",
    ").reset_index().sort_values('총생산량', ascending=False)\n",
    "by_model['양품률(%)'] = (by_model['합격'] / by_model['총생산량'] * 100).round(2)\n",
    "by_model['점유율(%)'] = (by_model['총생산량'] / by_model['총생산량'].sum() * 100).round(2)\n",
    "by_model['누적점유율(%)'] = by_model['점유율(%)'].cumsum().round(2)\n",
    "\n",
    "print('\\n[ 차종별 생산량 ]')\n",
    "display(by_model.style.format({\n",
    "    '총생산량': '{:,.0f}', '합격': '{:,.0f}', '불합격': '{:,.0f}',\n",
    "    '양품률(%)': '{:.2f}%', '점유율(%)': '{:.2f}%', '누적점유율(%)': '{:.2f}%'\n",
    "}))\n",
    "\n",
    "# 시각화\n",
    "brand_colors = {'HMC': '#003DA5', 'KIA': '#BB162B', 'GEN': '#9B7B56'}\n",
    "bar_colors = [brand_colors.get(b, 'gray') for b in by_model['brand']]\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(14, 6))\n",
    "bars = ax.bar(by_model['model_name'], by_model['총생산량'], color=bar_colors, alpha=0.85)\n",
    "for bar, val, share in zip(bars, by_model['총생산량'], by_model['점유율(%)']):\n",
    "    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3000,\n",
    "            f'{val:,.0f}\\n({share:.1f}%)', ha='center', fontsize=9, fontweight='bold')\n",
    "ax.set_ylabel('생산량 (대)', fontsize=12)\n",
    "ax.set_title('차종별 총 생산량', fontsize=14, fontweight='bold')\n",
    "ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/10000:.0f}만'))\n",
    "\n",
    "legend_handles = [Patch(facecolor=c, label=p) for p, c in brand_colors.items()]\n",
    "ax.legend(handles=legend_handles, title='브랜드', fontsize=10)\n",
    "ax.grid(axis='y', alpha=0.3)\n",
    "plt.xticks(rotation=30, ha='right')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 8. 브랜드별 생산량"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 8. 브랜드별 생산량 (HMC / KIA / GEN)\n",
    "# ========================================\n",
    "by_brand = daily.groupby('brand').agg(\n",
    "    총생산량=('total_inspections', 'sum'),\n",
    "    합격=('pass_count', 'sum'),\n",
    "    불합격=('fail_count', 'sum'),\n",
    "    모델수=('model_code', 'nunique')\n",
    ").reset_index().sort_values('총생산량', ascending=False)\n",
    "by_brand['양품률(%)'] = (by_brand['합격'] / by_brand['총생산량'] * 100).round(2)\n",
    "by_brand['점유율(%)'] = (by_brand['총생산량'] / by_brand['총생산량'].sum() * 100).round(2)\n",
    "\n",
    "brand_name_map = {'HMC': '현대자동차', 'KIA': '기아', 'GEN': '제네시스'}\n",
    "by_brand['브랜드명'] = by_brand['brand'].map(brand_name_map)\n",
    "\n",
    "print('\\n[ 브랜드별 생산량 ]')\n",
    "display(by_brand[['브랜드명', '모델수', '총생산량', '합격', '불합격', '양품률(%)', '점유율(%)']].style.format({\n",
    "    '총생산량': '{:,.0f}', '합격': '{:,.0f}', '불합격': '{:,.0f}',\n",
    "    '양품률(%)': '{:.2f}%', '점유율(%)': '{:.2f}%'\n",
    "}))\n",
    "\n",
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))\n",
    "colors = ['#003DA5', '#BB162B', '#9B7B56']\n",
    "ax1.pie(by_brand['총생산량'], labels=by_brand['브랜드명'], autopct='%1.1f%%',\n",
    "        colors=colors, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})\n",
    "ax1.set_title('브랜드별 생산 비중', fontsize=14, fontweight='bold')\n",
    "\n",
    "x = range(len(by_brand))\n",
    "bars = ax2.bar(x, by_brand['총생산량'], color=colors, alpha=0.85)\n",
    "ax2.set_xticks(x)\n",
    "ax2.set_xticklabels(by_brand['브랜드명'], fontsize=12)\n",
    "ax2.set_ylabel('생산량 (대)', fontsize=12)\n",
    "ax2.set_title('브랜드별 생산량', fontsize=14, fontweight='bold')\n",
    "ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/10000:.0f}만'))\n",
    "for bar, val in zip(bars, by_brand['총생산량']):\n",
    "    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,\n",
    "             f'{val:,.0f}', ha='center', fontsize=11, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 9. 교대조별 생산량 및 품질"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 9. 교대조별 생산량 및 품질\n",
    "# ========================================\n",
    "by_shift = daily.groupby('shift').agg(\n",
    "    총생산량=('total_inspections', 'sum'),\n",
    "    합격=('pass_count', 'sum'),\n",
    "    불합격=('fail_count', 'sum')\n",
    ").reset_index()\n",
    "by_shift['양품률(%)'] = (by_shift['합격'] / by_shift['총생산량'] * 100).round(2)\n",
    "by_shift['불량률(%)'] = (by_shift['불합격'] / by_shift['총생산량'] * 100).round(2)\n",
    "by_shift['점유율(%)'] = (by_shift['총생산량'] / by_shift['총생산량'].sum() * 100).round(2)\n",
    "\n",
    "shift_desc = {'A': 'A조 (주간 06~13시)', 'B': 'B조 (오후 14~21시)', 'C': 'C조 (야간 22시~)'}\n",
    "by_shift['교대조명'] = by_shift['shift'].map(shift_desc)\n",
    "\n",
    "print('\\n[ 교대조별 생산량 및 품질 ]')\n",
    "display(by_shift[['교대조명', '총생산량', '합격', '불합격', '양품률(%)', '불량률(%)', '점유율(%)']].style.format({\n",
    "    '총생산량': '{:,.0f}', '합격': '{:,.0f}', '불합격': '{:,.0f}',\n",
    "    '양품률(%)': '{:.2f}%', '불량률(%)': '{:.2f}%', '점유율(%)': '{:.2f}%'\n",
    "}).background_gradient(subset=['불량률(%)'], cmap='Reds'))\n",
    "\n",
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))\n",
    "shift_colors = ['#FFC107', '#FF9800', '#37474F']\n",
    "\n",
    "bars = ax1.bar(by_shift['교대조명'], by_shift['총생산량'], color=shift_colors, alpha=0.85)\n",
    "for bar, val in zip(bars, by_shift['총생산량']):\n",
    "    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,\n",
    "             f'{val:,.0f}', ha='center', fontsize=11, fontweight='bold')\n",
    "ax1.set_ylabel('생산량 (대)', fontsize=12)\n",
    "ax1.set_title('교대조별 생산량', fontsize=14, fontweight='bold')\n",
    "ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/10000:.0f}만'))\n",
    "\n",
    "bars2 = ax2.bar(by_shift['교대조명'], by_shift['불량률(%)'], color=shift_colors, alpha=0.85)\n",
    "for bar, val in zip(bars2, by_shift['불량률(%)']):\n",
    "    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,\n",
    "             f'{val:.2f}%', ha='center', fontsize=12, fontweight='bold', color='red')\n",
    "ax2.axhline(y=by_shift['불량률(%)'].mean(), color='gray', linestyle='--', alpha=0.7)\n",
    "ax2.set_ylabel('불량률 (%)', fontsize=12)\n",
    "ax2.set_title('교대조별 불량률 비교', fontsize=14, fontweight='bold')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 10. 요일별 생산량 패턴"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 10. 요일별 생산량 패턴\n",
    "# ========================================\n",
    "weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']\n",
    "weekday_kr = ['월', '화', '수', '목', '금', '토', '일']\n",
    "\n",
    "by_weekday = daily.groupby('weekday_name').agg(\n",
    "    총생산량=('total_inspections', 'sum'),\n",
    "    일수=('date', 'nunique')\n",
    ").reindex(weekday_order)\n",
    "by_weekday['일평균생산량'] = (by_weekday['총생산량'] / by_weekday['일수']).astype(int)\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(10, 5))\n",
    "colors = ['#2196F3'] * 5 + ['#FF5722'] * 2\n",
    "bars = ax.bar(weekday_kr, by_weekday['일평균생산량'], color=colors, alpha=0.85)\n",
    "for bar, val in zip(bars, by_weekday['일평균생산량']):\n",
    "    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,\n",
    "            f'{val:,}', ha='center', fontsize=11, fontweight='bold')\n",
    "ax.set_ylabel('일 평균 생산량 (대)', fontsize=12)\n",
    "ax.set_title('요일별 일 평균 생산량', fontsize=14, fontweight='bold')\n",
    "ax.grid(axis='y', alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print('\\n[ 요일별 생산 상세 ]')\n",
    "by_weekday.index = weekday_kr\n",
    "display(by_weekday.style.format({'총생산량': '{:,.0f}', '일평균생산량': '{:,.0f}'}))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 11. 차종별 월간 생산량 히트맵"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 11. 차종별 월간 생산량 히트맵\n",
    "# ========================================\n",
    "pivot = daily.groupby(['year_month', 'model_name'])['total_inspections'].sum().unstack(fill_value=0)\n",
    "\n",
    "# 생산량 상위 모델 순으로 정렬\n",
    "model_order = by_model.sort_values('총생산량', ascending=False)['model_name'].tolist()\n",
    "pivot = pivot[model_order]\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(18, 8))\n",
    "im = ax.imshow(pivot.T.values, aspect='auto', cmap='YlOrRd')\n",
    "\n",
    "ax.set_xticks(range(0, len(pivot), 3))\n",
    "ax.set_xticklabels(pivot.index.astype(str)[::3], rotation=45, ha='right', fontsize=8)\n",
    "ax.set_yticks(range(len(model_order)))\n",
    "ax.set_yticklabels(model_order, fontsize=10)\n",
    "ax.set_title('차종별 월간 생산량 히트맵', fontsize=14, fontweight='bold')\n",
    "\n",
    "cbar = plt.colorbar(im, ax=ax, shrink=0.8)\n",
    "cbar.set_label('생산량 (대)', fontsize=11)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 12. 공장 × 차종 크로스탭"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 12. 공장 × 차종 크로스탭\n",
    "# ========================================\n",
    "cross_plant_model = daily.groupby(['plant_name', 'model_name'])['total_inspections'].sum().unstack(fill_value=0)\n",
    "cross_plant_model = cross_plant_model[model_order]\n",
    "cross_plant_model['합계'] = cross_plant_model.sum(axis=1)\n",
    "\n",
    "print('\\n[ 공장 × 차종 생산량 크로스탭 ]')\n",
    "display(cross_plant_model.style.format('{:,.0f}').background_gradient(cmap='Blues', axis=None))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 13. 일별 생산량 분포"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 13. 일별 생산량 분포 (히스토그램 + 박스플롯)\n",
    "# ========================================\n",
    "daily_total = daily.groupby('date')['total_inspections'].sum()\n",
    "\n",
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "ax1.hist(daily_total, bins=50, color='#2196F3', alpha=0.7, edgecolor='white')\n",
    "ax1.axvline(daily_total.mean(), color='red', linestyle='--', label=f'평균: {daily_total.mean():,.0f}')\n",
    "ax1.axvline(daily_total.median(), color='green', linestyle='--', label=f'중위수: {daily_total.median():,.0f}')\n",
    "ax1.set_xlabel('일 생산량 (대)', fontsize=12)\n",
    "ax1.set_ylabel('빈도', fontsize=12)\n",
    "ax1.set_title('일별 생산량 분포', fontsize=14, fontweight='bold')\n",
    "ax1.legend(fontsize=10)\n",
    "\n",
    "ax2.boxplot(daily_total, vert=True, patch_artist=True,\n",
    "            boxprops=dict(facecolor='#2196F3', alpha=0.5))\n",
    "ax2.set_ylabel('일 생산량 (대)', fontsize=12)\n",
    "ax2.set_title('일별 생산량 박스플롯', fontsize=14, fontweight='bold')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(f'\\n일별 생산량 통계:')\n",
    "print(f'  평균: {daily_total.mean():,.0f}대  |  중위수: {daily_total.median():,.0f}대')\n",
    "print(f'  표준편차: {daily_total.std():,.0f}대')\n",
    "print(f'  최소: {daily_total.min():,.0f}대  |  최대: {daily_total.max():,.0f}대')\n",
    "print(f'  Q1: {daily_total.quantile(0.25):,.0f}대  |  Q3: {daily_total.quantile(0.75):,.0f}대')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 14. 종합 통계 요약 테이블"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ========================================\n",
    "# 14. 종합 통계 요약 테이블 (최종 정리)\n",
    "# ========================================\n",
    "print('\\n' + '=' * 70)\n",
    "print('         현대자동차 도장 검사 기반 생산량 종합 통계')\n",
    "print('=' * 70)\n",
    "\n",
    "print(f'''\n",
    "■ 분석 기간: {daily[\"date\"].min().date()} ~ {daily[\"date\"].max().date()} ({total_days}일)\n",
    "■ 데이터 규모: 검사 기록 3,000,000건 / 결함 기록 170,904건\n",
    "\n",
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n",
    "  [1] 전체 생산량\n",
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n",
    "  총 생산(검사) 건수 : {total:>12,}대\n",
    "  합격 (PASS)       : {total_pass:>12,}대 ({total_pass/total*100:.2f}%)\n",
    "  불합격 (FAIL)     : {total_fail:>12,}대 ({total_fail/total*100:.2f}%)\n",
    "  일 평균 생산량    : {total/total_days:>12,.0f}대/일\n",
    "''')\n",
    "\n",
    "print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')\n",
    "print('  [2] 공장별 생산량 (상위 순)')\n",
    "print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')\n",
    "for _, r in by_plant.iterrows():\n",
    "    print(f\"  {r['plant_name']:8s} ({r['plant_code']}) : {r['총생산량']:>10,.0f}대  \"\n",
    "          f\"(점유율 {r['점유율(%)']:.1f}%, 양품률 {r['양품률(%)']:.2f}%, {r['라인수']}개 라인)\")\n",
    "\n",
    "print(f'''\n",
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n",
    "  [3] 브랜드별 생산량\n",
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━''')\n",
    "for _, r in by_brand.iterrows():\n",
    "    print(f\"  {r['브랜드명']:10s} : {r['총생산량']:>10,.0f}대 ({r['점유율(%)']:.1f}%, {r['모델수']}개 모델)\")\n",
    "\n",
    "print(f'''\n",
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n",
    "  [4] 차종별 생산량 (Top 5)\n",
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━''')\n",
    "for i, (_, r) in enumerate(by_model.head(5).iterrows(), 1):\n",
    "    print(f\"  {i}위. {r['model_name']:10s} ({r['brand']}) : {r['총생산량']:>10,.0f}대 ({r['점유율(%)']:.1f}%)\")\n",
    "\n",
    "print(f'''\n",
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n",
    "  [5] 교대조별\n",
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━''')\n",
    "for _, r in by_shift.iterrows():\n",
    "    print(f\"  {r['교대조명']:20s} : {r['총생산량']:>10,.0f}대  (불량률 {r['불량률(%)']:.2f}%)\")\n",
    "\n",
    "print('\\n' + '=' * 70)\n",
    "print('  ※ 본 데이터는 도장 품질 검사(AI 비전 검사) 기록 기반이며,')\n",
    "print('    검사 건수 ≈ 생산 차체 수로 해석할 수 있음.')\n",
    "print('=' * 70)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}
